In [19]:
# Importazione moduli e librerie necessarie

import numpy as np # Funzioni matematiche e gestione array
import matplotlib.pyplot as plt # Plot
import os # Accedere al sistema operativo per la navigazione delle cartelle
import sys
import pandas as pd 
from pathlib import Path

In [25]:
# Creiamo gli input e gli output del codice: gli input sono l'array contenente le coordinate (x, y, z) a cui si trova il picco e i file delle simulazioni

coord_path = Path("C:\\Users\\plele\\OneDrive\\Desktop\\Università\\Magistrale\\Advanced Machine Learning\\IRIDE\\data\\Risultati")
sim_path = Path("C:\\Users\\plele\\OneDrive\\Desktop\\Università\\Magistrale\\Advanced Machine Learning\\IRIDE\\data\\Dose_3D")

dose_output = Path("C:\\Users\\plele\\OneDrive\\Desktop\\Università\\Magistrale\\Advanced Machine Learning\\IRIDE\\data\\Risultati")
regione_apice = Path("C:\\Users\\plele\\OneDrive\\Desktop\\Università\\Magistrale\\Advanced Machine Learning\\IRIDE\\data\\Risultati")

dose_output.mkdir(parents = True, exist_ok = True) # Se la cartella non esiste la crea

# Dichiariamo le variabili del codice

phi_fisso = 0.0
raggio_placca = 12.1
ATTIVITA_BQ = 10e6
N_RUN = 10
PRESC_TUMORE = 50 # Dose minima all'apice del tumore

# Creo il dizionario degli organi

ORGANI = {
    0: "Aria",
    1415: "Sclera",
    1417: "Cornea",
    1418: "Retina",
    1420: "Cristallino",
    2882: "Nervo Ottico",
    9000: "Umor Vitreo",
    9999: "Tumore",
}

# Creo una lista ordinata
# Ordine fisso degli ID degli organi per mantenere coerenti le colonne del tensore
# [Cristallino, Retina, Cornea, Nervo Ottico, Sclera]

ORDINE_ORGANI_OUTPUT = [1420, 1418, 1417, 2882, 1415]

In [24]:
PosPeak = coord_path / "apici_tumore_72_posizioni.txt"

# Definiamo la formattazione del file .npy

if PosPeak.exists():
    df_coord = pd.read_csv(
    PosPeak, sep=r'\s+', comment='#',
        names=['theta_gradi', 'x_apice_mm', 'y_apice_mm', 'z_apice_mm', 'extra']
    ).drop(columns=['extra'], errors='ignore')
else:
    print(f"Attenzione: Il file delle coordinate non esiste in {PosPeak}")
    df_coord = None

Raggio_sfera = 2.0 # mm = 4 voxel

lista_df_apice= []
# Ora creiamo la regione che definisce l'apice 
if df_coord is not None:
    for theta in range(0, 360, 5):
        
        theta_file = theta -360 if theta >= 315 else theta

        # Definiamo il centro della regione sferica che identifica l'apice sul tumore

        #if df_coord is not None:
        coord_corrente = df_coord[df_coord['theta_gradi'] == theta_file]

        if not coord_corrente.empty:
            # Recuperiamo le coordinate x, y, z del picco
            x_c = coord_corrente['x_apice_mm'].values[0]
            y_c = coord_corrente['y_apice_mm'].values[0]
            z_c = coord_corrente['z_apice_mm'].values[0]
        else:
            # Se non troviamo coordinate per questo specifico theta, saltiamo al prossimo
            print(f"Coordinate non trovate per theta = {theta_file}")
            continue

        nome_file_s1 = f"dose_voxel_theta{theta_file}_phi{int(phi_fisso)}_s1.txt"
        percorso_file_s1 = sim_path / nome_file_s1 # Dice dove andare a prendere i file   

        if percorso_file_s1.exists():
            df_dose = pd.read_csv(
                percorso_file_s1, sep='\t', comment='#',
                names=['voxelId', 'organId', 'x_mm', 'y_mm', 'z_mm', 'dose_Gy_per_event', '_extra']
            ).drop(columns=['_extra'], errors='ignore')

            # Creiamo la regione sferica 

            distanze = np.sqrt((df_dose['x_mm'] - x_c)**2 + (df_dose['y_mm'] - y_c)**2 + (df_dose['z_mm'] - z_c)**2)

            # Filtriamo i voxel: prendiamo solo i voxel tumore con distanza minore del raggio della regione  
            df_apice = df_dose[(distanze <= Raggio_sfera) & (df_dose['organId'] == 9999)].copy()
        
            # Salviamo il risultato parziale nella lista
            lista_df_apice.append(df_apice)
            print(f"Theta {theta_file}: trovati {len(df_apice)} voxel all'interno della sfera di raggio {Raggio_sfera}mm.")
if lista_df_apice:
    df_apice_totale = pd.concat(lista_df_apice, ignore_index=True)
    print(f"\nEstrazione completata con successo!")
    print(f"Voxel tumorali totali estratti (somma di tutti i theta): {len(df_apice_totale)}")
else:
    print("\nNessun dato estratto. Verifica i percorsi o la presenza dei file.")


Theta 0: trovati 98 voxel all'interno della sfera di raggio 2.0mm.
Theta 5: trovati 109 voxel all'interno della sfera di raggio 2.0mm.
Theta 10: trovati 109 voxel all'interno della sfera di raggio 2.0mm.
Theta 15: trovati 109 voxel all'interno della sfera di raggio 2.0mm.
Theta 20: trovati 109 voxel all'interno della sfera di raggio 2.0mm.
Theta 25: trovati 103 voxel all'interno della sfera di raggio 2.0mm.
Theta 30: trovati 103 voxel all'interno della sfera di raggio 2.0mm.
Theta 35: trovati 103 voxel all'interno della sfera di raggio 2.0mm.
Theta 40: trovati 103 voxel all'interno della sfera di raggio 2.0mm.
Theta 45: trovati 103 voxel all'interno della sfera di raggio 2.0mm.
Theta 50: trovati 113 voxel all'interno della sfera di raggio 2.0mm.
Theta 55: trovati 110 voxel all'interno della sfera di raggio 2.0mm.
Theta 60: trovati 110 voxel all'interno della sfera di raggio 2.0mm.
Theta 65: trovati 106 voxel all'interno della sfera di raggio 2.0mm.
Theta 70: trovati 106 voxel all'inter

In [27]:
file_regione = regione_apice / "regione_apice.txt"

df_apici = pd.DataFrame(df_apice_totale)

#df_apici = df_apici.sort_values("theta_gradi").reset_index(drop=True)

print("\n================ CONTROLLO APICI ================")
print(f"Numero righe apici: {len(df_apici)}")

if len(df_apici) != 72:
    print(f"ATTENZIONE: il file finale avrà {len(df_apici)} righe, non 72.")

display(df_apici)

df_apici.to_csv(
    file_regione,
    sep="\t",
    index=False,
    float_format="%.8f"
)

print(f"\nFile apici salvato in:\n{file_regione}")


================ CONTROLLO APICI ================
Numero righe apici: 7504
ATTENZIONE: il file finale avrà 7504 righe, non 72.


,voxelId,organId,x_mm,y_mm,z_mm,dose_Gy_per_event
0,62863,9999,6.75,-1.25,-6.25,9.065132e-12
1,62922,9999,6.25,-0.75,-6.25,5.899856e-12
2,62923,9999,6.75,-0.75,-6.25,9.840605e-12
3,62924,9999,7.25,-0.75,-6.25,1.620738e-11
4,62981,9999,5.75,-0.25,-6.25,4.586714e-12
...,...,...,...,...,...,...
7499,73786,9999,8.25,-0.25,-4.75,1.733445e-11
7500,73843,9999,6.75,0.25,-4.75,5.284571e-12
7501,73844,9999,7.25,0.25,-4.75,8.633973e-12
7502,73845,9999,7.75,0.25,-4.75,1.105179e-11



File apici salvato in:
C:\Users\plele\OneDrive\Desktop\Università\Magistrale\Advanced Machine Learning\IRIDE\data\Risultati\regione_apice.txt
